In [3]:
import seaborn as sns
df=sns.load_dataset('tips')

In [4]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [7]:
df['time'].value_counts()

time
Dinner    176
Lunch      68
Name: count, dtype: int64

##### Feature Encoding (Label Encoding And OneHot Encoding)

In [45]:
## Indepenedent and Dependent features
X = df.drop(['total_bill'],axis=1)
Y = df['total_bill']

In [46]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=2)

In [47]:
## Feature Encoding 
from sklearn.preprocessing import LabelEncoder
le1 = LabelEncoder()
le2 = LabelEncoder()
le3 = LabelEncoder()

In [48]:
X_train['sex']=le1.fit_transform(X_train['sex'])
X_train['smoker']=le2.fit_transform(X_train['smoker'])
X_train['time']=le3.fit_transform(X_train['time'])

In [49]:
X_train.head()

,tip,sex,smoker,day,time,size
144,2.3,0,0,Thur,1,2
154,2.0,1,0,Sun,0,4
184,3.0,1,1,Sun,0,2
161,2.5,1,0,Sun,0,2
44,5.6,1,0,Sun,0,4


In [50]:
X_test['sex']=le1.transform(X_test['sex'])
X_test['smoker']=le2.transform(X_test['smoker'])
X_test['time']=le3.transform(X_test['time'])

In [51]:
X_train['day'].value_counts()
## Days have 4 classes not a binary class so we will use OneHotEncoding for this column. We will use ColumnTransformer to do this.

day
Sat     69
Sun     61
Thur    51
Fri     14
Name: count, dtype: int64

In [52]:
## OneHotencoding -- COlumnTransformer

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first'), [3])
    ],
    remainder='passthrough'
)

In [53]:
import sys
import numpy as np 
np.set_printoptions(threshold=sys.maxsize)
X_test=ct.fit_transform(X_train)

In [54]:
X_test=ct.fit_transform(X_train)

In [61]:
X_train_encoded = ct.fit_transform(X_train)
X_test_original = X.loc[Y_test.index].copy()
X_test_original['sex'] = le1.transform(X_test_original['sex'])
X_test_original['smoker'] = le2.transform(X_test_original['smoker'])
X_test_original['time'] = le3.transform(X_test_original['time'])

X_test_encoded = ct.transform(X_test_original)

X_test_encoded[:5]

X_test_encoded[:5]

array([[0.  , 0.  , 1.  , 5.17, 0.  , 0.  , 1.  , 4.  ],
       [0.  , 1.  , 0.  , 4.34, 1.  , 0.  , 0.  , 4.  ],
       [0.  , 0.  , 1.  , 1.48, 1.  , 0.  , 1.  , 2.  ],
       [0.  , 0.  , 0.  , 4.3 , 0.  , 1.  , 0.  , 2.  ],
       [0.  , 1.  , 0.  , 2.55, 1.  , 0.  , 0.  , 2.  ]])

#### SVR -- Support Vector Regression

In [64]:
from sklearn.svm import SVR
svr=SVR(kernel='linear')
# SVR expects numeric features only; use the one-hot encoded arrays
svr.fit(X_train_encoded, Y_train)
y_pred = svr.predict(X_test_encoded)
y_pred=svr.predict(X_test)


In [66]:
from sklearn.metrics import mean_squared_error, r2_score
y_pred = svr.predict(X_test_encoded)  # X_test_encoded matches Y_test length
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)

print(f"MSE: {mse}")
print(f"R2 Score: {r2}")    


MSE: 27.038988793051733
R2 Score: 0.6548142333682279
